# Extraction of ECG-FM embeddings

Extracts the electrocardiogram foundation model (ECG-FM) embeddings used in the dominant frequency (DF) analysis (`02_DF_analysis.ipynb`).

- Model: ECG-FM (Hugging Face `wanglab/ecg-fm`, checkpoint `mimic_iv_ecg_physionet_pretrained.pt`; pretrained, not fine-tuned)
- Input: 12 leads, 500 Hz; each lead standardized and split into non-overlapping 5-s segments
- Embedding: Transformer output (`out['x']`, 768-dim) averaged over time and then across segments (used in the analysis)
- CNN-stage (256-dim) and first-5-s embeddings are also saved for reference (not used in the reported analyses)
- Saved in the same row order as the mapping table (`metadata.csv`), together with IDs, labels and file paths; alignment is checked in the analysis notebook

Order of execution: (1) this notebook → (2) `02_DF_analysis.ipynb`

Environment: Python 3.10, PyTorch, fairseq-signals, ecg-transform, huggingface_hub (see README for installation)


## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from huggingface_hub import hf_hub_download
from fairseq_signals.models import build_model_from_checkpoint

from ecg_transform.inp import ECGInput, ECGInputSchema
from ecg_transform.sample import ECGSample, ECGMetadata
from ecg_transform.t.common import HandleConstantLeads, LinearResample, ReorderLeads
from ecg_transform.t.scale import Standardize
from ecg_transform.t.cut import SegmentNonoverlapping


In [ ]:
ROOT    = "path/to/project"                 # <- change to your environment (same ROOT as the analysis notebook)
EMB_DIR = os.path.join(ROOT, "ecgfm")       # output folder for the embeddings
DEVICE  = 'cpu'                             # 'cuda' if a GPU is available
os.makedirs(EMB_DIR, exist_ok=True)

hf_hub_download(                            # download the model checkpoint (skipped if already present)
    repo_id='wanglab/ecg-fm',
    filename='mimic_iv_ecg_physionet_pretrained.pt',
    local_dir=os.path.join(ROOT, 'ckpts'),
)
CKPT_PATH = os.path.join(ROOT, 'ckpts', 'mimic_iv_ecg_physionet_pretrained.pt')

model = build_model_from_checkpoint(checkpoint_path=CKPT_PATH)
model.eval()
model.to(DEVICE)
print("Model loaded")


## 2. Input specification and extraction functions

In [ ]:
# Lead order = column names of the ECG CSV files
# (in our data, leads I, II and III are written with full-width Roman numerals; edit to match your files)
ECG_FM_LEAD_ORDER = ['Ⅰ', 'Ⅱ', 'Ⅲ', 'aVR', 'aVL', 'aVF',
                     'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
SAMPLE_RATE = 500
N_SAMPLES   = SAMPLE_RATE * 5          # 5 s (2500 samples)

ECG_FM_SCHEMA = ECGInputSchema(
    sample_rate=SAMPLE_RATE,
    expected_lead_order=ECG_FM_LEAD_ORDER,
    required_num_samples=N_SAMPLES,
)
ECG_FM_TRANSFORMS = [
    ReorderLeads(expected_order=ECG_FM_LEAD_ORDER, missing_lead_strategy='zero'),
    LinearResample(desired_sample_rate=SAMPLE_RATE),
    HandleConstantLeads(strategy='zero'),
    Standardize(),
    SegmentNonoverlapping(segment_length=N_SAMPLES),
]


In [ ]:
def encoder_out_to_emb(x):
    # average over time, excluding padding (0)
    return torch.div(x.sum(dim=1), (x != 0).sum(dim=1))

def run_model(feats):
    """Waveform (12, n) -> Transformer (768) and CNN-stage (256) embeddings, averaged across segments"""
    n_samples = feats.shape[1]
    meta = ECGMetadata(sample_rate=SAMPLE_RATE, num_samples=n_samples,
                       lead_names=ECG_FM_LEAD_ORDER, unit=None,
                       input_start=0, input_end=n_samples)
    sample = ECGSample(ECGInput(feats, meta), ECG_FM_SCHEMA, ECG_FM_TRANSFORMS)
    source = torch.from_numpy(sample.out).float().to(DEVICE)
    with torch.no_grad():
        out = model(source=source, features_only=True, mask=False)
        emb_t = encoder_out_to_emb(out['x']).mean(dim=0)          # 768
        emb_c = encoder_out_to_emb(out['features']).mean(dim=0)   # 256
    return emb_t.cpu().numpy(), emb_c.cpu().numpy()

def extract(csv_path):
    """Returns embeddings of the full record (main) and the first 5 s (reference)"""
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    df.columns = df.columns.str.strip()
    feats = df[ECG_FM_LEAD_ORDER].values.T.astype(np.float32)
    t_full, c_full   = run_model(feats)
    t_first, c_first = run_model(feats[:, :N_SAMPLES])
    return t_full, c_full, t_first, c_first


## 3. Extract and save for all recordings

In [ ]:
metadata = pd.read_csv(os.path.join(ROOT, "data/metadata.csv"), encoding="utf-8-sig")
metadata.columns = metadata.columns.str.strip()
metadata = metadata.dropna(subset=['ecg_path', 'ID', 'Result']).reset_index(drop=True)
print(f"{len(metadata)} ECGs / {metadata['ID'].nunique()} patients")

res = [extract(p) for p in tqdm(metadata['ecg_path'], desc="ECG-FM")]
emb_t_full  = np.array([r[0] for r in res])   # (n, 768) full record, Transformer output [used in the analysis]
emb_c_full  = np.array([r[1] for r in res])   # (n, 256) full record, CNN stage [reference]
emb_t_first = np.array([r[2] for r in res])   # (n, 768) first 5 s, Transformer output [reference]
emb_c_first = np.array([r[3] for r in res])   # (n, 256) first 5 s, CNN stage [reference]

np.save(os.path.join(EMB_DIR, 'emb_transformer_full.npy'),    emb_t_full)
np.save(os.path.join(EMB_DIR, 'emb_cnn_full.npy'),            emb_c_full)
np.save(os.path.join(EMB_DIR, 'emb_transformer_first5s.npy'), emb_t_first)
np.save(os.path.join(EMB_DIR, 'emb_cnn_first5s.npy'),         emb_c_first)
# for the alignment check in the analysis notebook
np.save(os.path.join(EMB_DIR, 'labels.npy'),      metadata['Result'].values.astype(int))
np.save(os.path.join(EMB_DIR, 'patient_ids.npy'), metadata['ID'].values)
np.save(os.path.join(EMB_DIR, 'ecg_paths.npy'),   metadata['ecg_path'].values)

print("Saved: full record, Transformer output", emb_t_full.shape)
print("Patients:", metadata['ID'].nunique())


## 4. Software versions

In [ ]:
import sys
import fairseq_signals
print(f"Python {sys.version.split()[0]} / numpy {np.__version__} / pandas {pd.__version__} / "
      f"PyTorch {torch.__version__} / fairseq-signals {getattr(fairseq_signals, '__version__', 'n/a')}")
